[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jrpmichel/Libro_Regularizacion_La_Salle_Bajio/blob/main/alg/u1_vectores.ipynb)

# Álg. U1 · Vectores, rectas y planos
**Libro de regularización en matemáticas para ingeniería · ULSA Bajío**

Este notebook acompaña la unidad 1 del bloque de álgebra lineal. Tiene una sección por subtema, en el mismo orden que el libro:

1. Vectores en 2D y 3D: componentes y magnitud
2. Suma, resta y vector unitario
3. Producto punto y proyección
4. Producto cruz
5. Rectas y planos en el espacio

**Cómo usarlo.** Ejecuta las celdas en orden (Entorno de ejecución → Ejecutar todas). Cada sección tiene una calculadora, sus casos de prueba y termina con una celda **Contrasta**: antes de ejecutarla, resuelve a mano el caso que se indica y escribe tu resultado. La referencia de la calculadora aparece solo después de que escribes el tuyo.

**Etiqueta del repositorio.** `ALG-U1` (bloque, unidad). Las celdas de código de cada sección vienen de `alg/u1_vectores/NN_*.py`; los fragmentos que el libro imprime, de `alg/u1_vectores/fragmentos/`.

**Convenciones.** Punto decimal. Un vector se escribe como números separados por comas, por ejemplo `1, -2, 3`; si quieres, enciérralo entre paréntesis o corchetes: `(1, -2, 3)` o `[1 -2 3]` (sin comas, separa con espacios). Cada componente acepta fracciones como `2/3`, raíces como `sqrt(2)` y `pi`. Los ángulos salen en **grados** o **radianes** según el control `unidad`, y la unidad siempre aparece junto al valor. Los valores se *redondean* (no se truncan) a las cifras significativas que elijas; los muy grandes o muy pequeños se escriben en notación científica, por ejemplo `1.234×10^-7`.

## 0. Versiones e imports

In [ ]:
# Versiones mínimas probadas. En Colab ya vienen instaladas; esta celda solo lo asegura.
%pip install -q "numpy>=1.26" "sympy>=1.12" "matplotlib>=3.8" "ipywidgets>=8"

import sys, math, re
from decimal import Decimal, ROUND_HALF_UP
from fractions import Fraction
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401  (registra la proyección "3d")
import sympy as sp
from sympy.core.evalf import PrecisionExhausted
import ipywidgets as widgets
from IPython.display import display
from sympy.parsing.sympy_parser import (parse_expr, standard_transformations,
                                        implicit_multiplication_application, convert_xor, rationalize)

print("Python", sys.version.split()[0], "| numpy", np.__version__, "| sympy", sp.__version__,
      "| matplotlib", matplotlib.__version__, "| ipywidgets", widgets.__version__)

In [ ]:
# Utilidades compartidas por todo el notebook
_PERMITIDOS = {"sqrt": sp.sqrt, "pi": sp.pi}
_GLOBAL = {"Integer": sp.Integer, "Float": sp.Float, "Rational": sp.Rational,
           "Add": sp.Add, "Mul": sp.Mul, "Pow": sp.Pow}
# rationalize: 0.1 se lee como 1/10, para que las fracciones salgan exactas antes de pasar a decimal
_TRANSF = standard_transformations + (implicit_multiplication_application, convert_xor, rationalize)
_CARACTERES = re.compile(r"^[0-9A-Za-z.+\-*/^() ]+$")
_LITERAL = re.compile(r"\d*\.\d+(?:[eE][+-]?\d+)?|\d+\.?(?:[eE][+-]?\d+)?")
_MAXIMO, _MINIMO = 1e100, 1e-100     # fuera de este rango los productos desbordan el punto flotante
_CIERRE = {"(": ")", "[": "]", "<": ">", "⟨": "⟩"}


def _contrae(preposicion, nombre):
    """'de' + 'el vector' -> 'del vector';  'a' + 'el valor' -> 'al valor'."""
    return f"{preposicion}l {nombre[3:]}" if nombre.startswith("el ") else f"{preposicion} {nombre}"


def _revisa_rango(v, nombre):
    """Rechaza componentes tan grandes o tan pequeñas que los productos se desbordan."""
    if v != 0 and not (_MINIMO * (1 - 1e-9) <= abs(v) <= _MAXIMO * (1 + 1e-9)):
        raise ValueError(f"{nombre} tiene un número fuera del rango 1e-100 a 1e100 (en valor absoluto); "
                         "cambia de unidades, por ejemplo de mm a m.")


def _potencia_excesiva(p):
    """True si p es una potencia con exponente mayor que 1000 o con valor mayor que 10^2000:
    evaluarla de forma exacta tardaría demasiado (9^9^9)."""
    if not isinstance(p, sp.Pow):
        return False
    if not abs(sp.N(p.exp, 15)) <= 1000:
        return True
    v = sp.N(p, 15)
    return bool(v.is_finite) and abs(v) > sp.Integer(10)**2000


def _lee_numero(texto, nombre):
    """'2/3', 'sqrt(2)', '-1.5e3', '3pi' -> float, o un mensaje claro si algo no es un número real."""
    t = texto.strip().replace("−", "-").replace("–", "-").replace("π", "pi")
    t = re.sub(r"√\s*([\d.]+)", r"sqrt(\1)", t).replace("√", "sqrt")
    if not t:
        raise ValueError(f"{_contrae('a', nombre)} le falta un número: revisa que no sobren comas.")
    if not _CARACTERES.match(t):
        raise ValueError(f"en {nombre} hay un símbolo que no reconozco ({t!r}): usa solo números con punto "
                         "decimal, + - * / ^, paréntesis, sqrt(...) y pi.")
    if re.search(r"[\d.]\s+[\d.]", t):
        raise ValueError(f"en {nombre}, {t!r} tiene dos números separados solo por un espacio: "
                         "si son dos componentes, sepáralas con una coma.")
    if re.search(r"\.\d*\.", t):
        raise ValueError(f"en {nombre}, {t!r} tiene dos puntos decimales en un mismo número.")
    desconocidos = sorted(set(re.findall(r"[A-Za-z]\w*", _LITERAL.sub(" ", t))) - set(_PERMITIDOS))
    if desconocidos:
        raise ValueError(f"en {nombre} no reconozco {', '.join(desconocidos)}: escribe solo números, "
                         "fracciones como 2/3, sqrt(2) o pi.")
    try:
        expr = parse_expr(t, local_dict=dict(_PERMITIDOS), global_dict=dict(_GLOBAL),
                          transformations=_TRANSF, evaluate=False)
        # 9^9^9 no se evalúa (tardaría horas): se revisa cada potencia, de adentro hacia afuera
        grande = any(_potencia_excesiva(p) for p in sp.postorder_traversal(expr))
        valor = None
        if not grande:
            try:
                valor = sp.N(expr, 30, strict=True)
            except PrecisionExhausted:            # 4 - 4, pi - pi, 1/(1 - 1): se cancela; se evalúa exacto
                valor = sp.N(expr.doit(), 30)
    except Exception as err:
        raise ValueError(f"no pude leer {t!r} en {nombre}: usa * para multiplicar y revisa los paréntesis.") from err
    if grande:
        raise ValueError(f"en {nombre}, {t!r} tiene una potencia demasiado grande.")
    if not isinstance(valor, sp.Expr) or not valor.is_number or valor.has(sp.zoo, sp.nan, sp.oo, -sp.oo):
        raise ValueError(f"en {nombre}, {t!r} no da un número finito (¿dividiste entre 0?).")
    if sp.im(valor) != 0:
        raise ValueError(f"en {nombre}, {t!r} no es un número real (¿raíz de un negativo?).")
    valor = sp.re(valor)
    _revisa_rango(valor, nombre)
    return float(valor)


def numero(valor, nombre="el valor"):
    """Un número escrito como texto ('7', 'sqrt(49)', '2/3') o ya como número -> float."""
    if isinstance(valor, str):
        return _lee_numero(valor, nombre)
    if valor is None or isinstance(valor, (bool, np.bool_)):
        raise ValueError(f"{nombre} debe ser un número.")
    try:
        v = float(valor)
    except (TypeError, ValueError) as err:
        raise ValueError(f"{nombre} debe ser un número real.") from err
    if not math.isfinite(v):
        raise ValueError(f"{nombre} debe ser un número finito.")
    return v


def _sin_envoltura(t):
    """Quita un par de paréntesis o corchetes que encierra todo el texto: '(1, 2)' -> '1, 2'."""
    if len(t) < 2 or t[0] not in _CIERRE or t[-1] != _CIERRE[t[0]]:
        return t
    nivel = 0
    for i, ch in enumerate(t):
        nivel += (ch == t[0]) - (ch == _CIERRE[t[0]])
        if nivel == 0 and i < len(t) - 1:      # el primer paréntesis cierra antes del final: no envuelve todo
            return t
    return t[1:-1].strip()


def _partes(texto, nombre):
    t = texto.strip()
    while _sin_envoltura(t) != t:
        t = _sin_envoltura(t)
    if not t:
        raise ValueError(f"escribe {nombre} como números separados por comas, por ejemplo: 1, -2, 3")
    if t.count("(") != t.count(")"):
        raise ValueError(f"en {nombre} los paréntesis no cierran: revisa que cada ( tenga su ).")
    partes = [p.strip() for p in re.split(r"[,;]", t)] if re.search(r"[,;]", t) else t.split()
    if any(not p for p in partes):
        raise ValueError(f"en {nombre} sobra una coma o falta un número; escribe, por ejemplo: 1, -2, 3")
    return partes


def vector(texto, dim=None, nombre="el vector"):
    """'1, -2, 3', '(1,2,3)', '[1 2 3]', '2/3, sqrt(2)' o una lista de números -> arreglo de numpy (float).
    Con dim, exige esa cantidad de componentes."""
    if isinstance(texto, str):
        partes = _partes(texto, nombre)
    elif isinstance(texto, (list, tuple, np.ndarray)):
        partes = list(np.ravel(np.array(texto, dtype=object)))
    else:
        raise ValueError(f"escribe {nombre} como texto, por ejemplo: \"1, -2, 3\"")
    v = np.array([numero(p, nombre if isinstance(p, str) else f"cada componente {_contrae('de', nombre)}")
                  for p in partes], dtype=float)
    if dim is not None and len(v) != dim:
        raise ValueError(f"{nombre} debe tener {dim} componentes y tiene {len(v)}.")
    if len(v) < 2:
        raise ValueError(f"{nombre} necesita al menos dos componentes separadas por comas, por ejemplo: 1, -2, 3")
    return v


def cifras(valor, n=4):
    """Redondea (no trunca) a n cifras significativas con la regla escolar (5 sube) y conserva los ceros finales.
    Con exponente decimal mayor que 15 o menor que -4 usa notación científica: 1.234×10^-7."""
    v = float(valor)
    if math.isnan(v):
        return "no definido"
    if math.isinf(v):
        return "+∞" if v > 0 else "-∞"
    if v == 0:
        return "0"
    d = Decimal(repr(v))
    q = d.quantize(Decimal(1).scaleb(d.adjusted() - n + 1), rounding=ROUND_HALF_UP)
    if q.adjusted() != d.adjusted():          # 9.996 con 3 cifras pasa a 10.0: se reajusta la posición
        q = d.quantize(Decimal(1).scaleb(q.adjusted() - n + 1), rounding=ROUND_HALF_UP)
    e = q.adjusted()
    if e > 15 or e < -4:                      # muy grande o muy pequeño: mantisa × 10^e
        return f"{format(q.scaleb(-e), 'f')}×10^{e}"
    if e >= n:                                # enteros grandes: sin notación científica
        return f"{int(q):d}"
    return format(q, "f")


def fmt(v, n=4):
    """Vector como texto '(a, b, c)', cada componente redondeada a n cifras significativas."""
    return "(" + ", ".join(cifras(c, n) for c in np.ravel(v)) + ")"


def cerca(p, q, tol=1e-9):
    return abs(float(p) - float(q)) < tol


def _rechaza(fn):
    """True si fn() lanza ValueError (para probar que una calculadora rechaza una entrada)."""
    try:
        fn()
    except ValueError:
        return True
    return False


def en_unidad(rad, unidad):
    """Ángulo en radianes -> en la unidad pedida ('grados' o 'radianes')."""
    if unidad not in ("grados", "radianes"):
        raise ValueError("La unidad debe ser 'grados' o 'radianes'.")
    return math.degrees(rad) if unidad == "grados" else float(rad)


def de_unidad(valor, unidad):
    """Ángulo en 'grados' o 'radianes' -> en radianes."""
    if unidad not in ("grados", "radianes"):
        raise ValueError("La unidad debe ser 'grados' o 'radianes'.")
    return math.radians(valor) if unidad == "grados" else float(valor)


def _sufijo(unidad):
    return "°" if unidad == "grados" else " rad"


# --- cálculo interno (sin validar: lo usan las funciones públicas después de validar) ---
def _norma(v):
    return math.hypot(*v)                     # hypot no se desborda al elevar al cuadrado


def _punto(a, b):
    """a · b con la cancelación por redondeo limpiada: 1e-17 que debería ser 0 se vuelve 0."""
    productos = [float(x) * float(y) for x, y in zip(a, b)]
    s = math.fsum(productos)
    return 0.0 if abs(s) <= 1e-14 * math.fsum(abs(p) for p in productos) else s


def _limpia(valores, escalas):
    """Pone en 0 cada componente que solo es ruido de redondeo frente a su escala."""
    v = np.array(valores, dtype=float)
    v[np.abs(v) <= 1e-14 * np.asarray(escalas, dtype=float)] = 0.0
    return v + 0.0                            # + 0.0 cambia -0.0 por 0.0


def _escalado(v):
    """v multiplicado por una potencia de 2 (exacto) para que su mayor componente quede entre 0.5 y 1."""
    m = float(np.max(np.abs(v)))
    return v if m == 0 else np.ldexp(v, -math.frexp(m)[1])


def _par(a, b):
    """Lee a y b y exige que tengan la misma cantidad de componentes."""
    a, b = vector(a, nombre="a"), vector(b, nombre="b")
    if len(a) != len(b):
        raise ValueError(f"a y b deben tener la misma cantidad de componentes; tienen {len(a)} y {len(b)}.")
    return a, b


def _ejes_iguales(ax, puntos, margen=0.15):
    """Misma escala en todos los ejes (2D o 3D) para que los ángulos no se deformen."""
    P = np.atleast_2d(np.array(puntos, dtype=float))
    bajo, alto = P.min(axis=0), P.max(axis=0)
    centro = (bajo + alto) / 2
    radio = (alto - bajo).max() / 2 * (1 + margen)
    if not radio > 0:
        radio = max(1.0, 0.1 * np.abs(centro).max())
    lims = [(c - radio, c + radio) for c in centro]
    ax.set_xlim(*lims[0]); ax.set_ylim(*lims[1])
    if P.shape[1] == 3:
        ax.set_zlim(*lims[2]); ax.set_box_aspect((1, 1, 1))
    else:
        ax.set_aspect("equal")


def _vista(ax, vectores=(), normal=None):
    """Elige la vista 3D (elevación de 15° a 40°) en la que cada vector queda a 30° o más de la línea de
    visión y, si hay normal, su plano se ve inclinado (de 30° a 60° respecto a verlo de frente). Entre las
    vistas que lo cumplen, la más cercana a la de matplotlib por omisión (azimut -60°, elevación 30°)."""
    def unitario_o_nada(v):
        v = np.asarray(v, dtype=float)
        return v / _norma(v) if np.any(v) and np.all(np.isfinite(v)) else None
    dirs = [d for d in map(unitario_o_nada, vectores) if d is not None]
    nn = None if normal is None else unitario_o_nada(normal)
    mejor = None
    for elev in range(15, 41, 5):
        for azim in range(-180, 180, 5):
            e, a = math.radians(elev), math.radians(azim)
            ojo = np.array([math.cos(e) * math.cos(a), math.cos(e) * math.sin(a), math.sin(e)])
            senos = [math.sqrt(max(0.0, 1 - (ojo @ d) ** 2)) for d in dirs]
            if nn is not None:
                senos += [math.sqrt(max(0.0, 1 - (ojo @ nn) ** 2)), abs(ojo @ nn)]
            lejania = abs(elev - 30) + abs((azim + 60 + 180) % 360 - 180) / 4
            clave = (min(round(min(senos, default=1.0), 2), 0.5), -lejania)   # sen 30° = 0.5 basta
            if mejor is None or clave > mejor[0]:
                mejor = (clave, elev, azim)
    ax.view_init(elev=mejor[1], azim=mejor[2])

## 1. Vectores en 2D y 3D: componentes y magnitud

El vector que va del punto $P$ al punto $Q$ se obtiene restando **llegada menos salida**:

$$\overrightarrow{PQ}=Q-P=(q_x-p_x,\;q_y-p_y,\;q_z-p_z).$$

Su magnitud (norma) es su longitud, por Pitágoras (en el espacio, Pitágoras se aplica dos veces: primero en el piso y luego en la vertical):

$$\|\mathbf v\|=\sqrt{v_x^2+v_y^2}\quad\text{(plano)},\qquad \|\mathbf v\|=\sqrt{v_x^2+v_y^2+v_z^2}\quad\text{(espacio)}.$$

En el plano, el ángulo $\theta$ que forma $\mathbf v$ con el eje $x$ cumple $\tan\theta=v_y/v_x$, pero despejar $\theta=\arctan(v_y/v_x)$ **pierde el cuadrante**: $(3,4)$ y $(-3,-4)$ dan el mismo cociente y apuntan en sentidos opuestos. La función de dos argumentos `arctan2(vy, vx)` mira el signo de cada componente y devuelve un ángulo en $(-180^\circ,\,180^\circ]$, o en $(-\pi,\,\pi]$ si trabajas en radianes. El vector cero no tiene dirección, así que no tiene ángulo.

La calculadora recibe $P$ y $Q$ (2 o 3 componentes), escribe $\overrightarrow{PQ}$ y su magnitud y, en el plano, el ángulo con la unidad a la vista. Dibuja la flecha con sus componentes punteadas.

In [ ]:
def componentes(P, Q):
    """Vector de P a Q (llegada menos salida). P y Q deben tener 2 o 3 componentes, las mismas los dos."""
    P, Q = vector(P, nombre="P"), vector(Q, nombre="Q")
    if len(P) != len(Q) or len(P) not in (2, 3):
        raise ValueError(f"P y Q deben tener la misma cantidad de componentes, 2 (plano) o 3 (espacio); "
                         f"tienen {len(P)} y {len(Q)}.")
    return Q - P


def magnitud(v):
    """Longitud del vector: raíz de la suma de los cuadrados de sus componentes."""
    return _norma(vector(v))


def angulo_2d(v, unidad="grados"):
    """Ángulo de un vector del plano con el eje x, con arctan2: en (-180°, 180°] o (-π, π]."""
    v = vector(v)
    if len(v) != 2:
        raise ValueError("el ángulo con el eje x se calcula aquí solo para vectores del plano (2 componentes).")
    if not np.any(v):
        raise ValueError("el vector cero no tiene dirección.")
    t = math.atan2(v[1], v[0])
    if t == -math.pi:                         # (-1, -0.0) daría -180°: se reporta como 180°
        t = math.pi
    return en_unidad(t, unidad)


def calculadora_componentes(P_txt, Q_txt, unidad, n_cifras):
    try:
        P, Q = vector(P_txt, nombre="P"), vector(Q_txt, nombre="Q")
        v = componentes(P, Q)
    except ValueError as err:
        print("Revisa la entrada:", err); return
    cuadrados = " + ".join(f"{c}²" for c in ("vx", "vy", "vz")[:len(v)])
    print(f"P = {fmt(P, n_cifras)}    Q = {fmt(Q, n_cifras)}")
    print(f"PQ = Q − P = {fmt(v, n_cifras)}      (llegada menos salida)")
    print(f"‖PQ‖ = √({cuadrados}) = √{cifras(math.fsum(v**2), n_cifras)} = {cifras(magnitud(v), n_cifras)}")
    if not np.any(v):
        print("P = Q: PQ es el vector cero, que no tiene dirección (ni ángulo ni vector unitario).")
    elif len(v) == 2:
        try:
            ang = angulo_2d(v, unidad)
            print(f"ángulo con el eje x (arctan2) = {cifras(ang, n_cifras)}{_sufijo(unidad)}")
            if v[0] < 0:
                malo = en_unidad(math.atan(v[1] / v[0]), unidad)
                print(f"Ojo: arctan(vy/vx) daría {cifras(malo, n_cifras)}{_sufijo(unidad)}, el ángulo del vector "
                      "opuesto: arctan pierde el cuadrante.")
        except ValueError as err:
            print("ángulo:", err)
    else:
        print("En 3D un solo ángulo no basta para fijar la dirección; la da el vector unitario (sección 2).")
    with np.errstate(all="ignore"):          # componentes cercanas a 1e100: matplotlib eleva al cuadrado
        _dibuja_pq(P, Q, v)


def _dibuja_pq(P, Q, v):
    fig = plt.figure(figsize=(5.5, 4.5))
    if len(v) == 2:
        ax = fig.add_subplot()
        ax.plot([P[0], Q[0]], [P[1], P[1]], ":", color="tab:red", lw=1.5, label=f"componente x = {cifras(v[0], 3)}")
        ax.plot([Q[0], Q[0]], [P[1], Q[1]], ":", color="tab:green", lw=1.5, label=f"componente y = {cifras(v[1], 3)}")
        ax.quiver(*P, *v, angles="xy", scale_units="xy", scale=1, color="navy", width=0.012)
        ax.legend(loc="best", fontsize=8); ax.grid(True)
    else:
        ax = fig.add_subplot(projection="3d")
        esquinas = np.array([P, (Q[0], P[1], P[2]), (Q[0], Q[1], P[2]), Q])
        ax.plot(*esquinas.T, "k:", lw=1.2, label="componentes x, y, z")
        ax.quiver(*P, *v, color="navy", arrow_length_ratio=0.12, lw=2)
        ax.set_zlabel("z"); ax.legend(loc="upper left", fontsize=8)
        _vista(ax, [v])                        # que la flecha no apunte hacia el ojo
    for X, e in (((P, "P = Q"),) if not np.any(v) else ((P, "P"), (Q, "Q"))):
        ax.scatter(*X, color="black", s=15)
        ax.text(*X, f"  {e}", fontsize=11)
    _ejes_iguales(ax, [P, Q])
    ax.set_xlabel("x"); ax.set_ylabel("y")
    ax.set_title(f"PQ = {fmt(v, 3)},  ‖PQ‖ = {cifras(magnitud(v), 3)}")
    plt.show()


widgets.interact(calculadora_componentes,
    P_txt=widgets.Text(value="1, 1", description="P =", continuous_update=False),
    Q_txt=widgets.Text(value="4, 5", description="Q =", continuous_update=False),
    unidad=widgets.Dropdown(options=["grados", "radianes"], value="grados", description="unidad"),
    n_cifras=widgets.IntSlider(value=4, min=2, max=8, description="cifras sig.", continuous_update=False));

In [ ]:
# Casos de prueba de la sección 1 (resultado conocido)
PRUEBAS_1 = [
    ("PQ de (1, 1) a (4, 5) = (3, 4), magnitud 5, ángulo 53.13°",
     lambda: np.allclose(componentes((1, 1), (4, 5)), (3, 4)) and cerca(magnitud(componentes((1, 1), (4, 5))), 5)
             and round(angulo_2d(componentes((1, 1), (4, 5)), "grados"), 2) == 53.13),
    ("PQ de (1, -2, 4) a (3, 4, 1) = (2, 6, -3), magnitud 7",
     lambda: np.allclose(componentes((1, -2, 4), (3, 4, 1)), (2, 6, -3))
             and cerca(magnitud(componentes((1, -2, 4), (3, 4, 1))), 7)),
    ("ángulo de (-3, -4) = -126.87° (no 53.13°)", lambda: round(angulo_2d((-3, -4), "grados"), 2) == -126.87),
    ("el vector (0, 0) no tiene ángulo", lambda: _rechaza(lambda: angulo_2d((0, 0)))),
    ("entradas mal escritas se rechazan: '1, a' y vacía",
     lambda: _rechaza(lambda: vector("1, a")) and _rechaza(lambda: vector(""))),
]
for nombre, prueba in PRUEBAS_1:
    print("OK  " if prueba() else "FALLA", nombre)

**Contrasta (sección 1).** Calcula a mano $\|\overrightarrow{PQ}\|$ para $P(-1,2,3)$ y $Q(1,-1,9)$: primero las componentes (llegada menos salida), después la raíz de la suma de los cuadrados. Escribe tu resultado.

In [ ]:
mi_magnitud = None     # escribe tu resultado, por ejemplo: 5.385 o "sqrt(29)"

if mi_magnitud is None:     # la referencia se muestra después de tu cálculo
    print("falta tu cálculo: escríbelo arriba y vuelve a ejecutar la celda")
else:
    v_ref = componentes((-1, 2, 3), (1, -1, 9))
    print("La calculadora da: PQ =", fmt(v_ref, 4), "  ‖PQ‖ =", cifras(magnitud(v_ref), 4))
    try:
        ok = cerca(numero(mi_magnitud, "tu resultado"), magnitud(v_ref), 1e-3)
    except ValueError as err:
        print("Revisa tu valor:", err); ok = None
    if ok is not None:
        print("coinciden" if ok else
              "NO coinciden: revisa las restas (llegada menos salida) y eleva al cuadrado también las componentes negativas")

## 2. Suma, resta y vector unitario

* **Suma** componente a componente: $\mathbf a+\mathbf b=(a_x+b_x,\;a_y+b_y,\;a_z+b_z)$. Solo se suman vectores con la misma cantidad de componentes. La **resta** es $\mathbf a-\mathbf b=\mathbf a+(-1)\,\mathbf b$.
* **Múltiplo escalar:** $k\mathbf a=(ka_x,\;ka_y,\;ka_z)$ y $\|k\mathbf a\|=|k|\,\|\mathbf a\|$. Si $k<0$, el sentido se invierte.
* **Vector unitario:** $\hat{\mathbf u}=\dfrac{\mathbf v}{\|\mathbf v\|}$ para $\mathbf v\neq\mathbf 0$. Tiene la misma dirección y sentido que $\mathbf v$ y magnitud 1. El vector cero no tiene vector unitario.
* **Desigualdad del triángulo:** $\|\mathbf a+\mathbf b\|\le\|\mathbf a\|+\|\mathbf b\|$. La igualdad vale solo si los dos apuntan en el mismo sentido (o uno es cero). Por eso dos fuerzas perpendiculares de 300 N y 400 N dan una resultante de 500 N, no de 700 N.

La calculadora forma $\alpha\mathbf a+\beta\mathbf b$ (con $\alpha=1$, $\beta=-1$ obtienes la resta), da su magnitud y su vector unitario, y compara $\|\alpha\mathbf a+\beta\mathbf b\|$ con $|\alpha|\,\|\mathbf a\|+|\beta|\,\|\mathbf b\|$. En el plano dibuja el paralelogramo.

In [ ]:
def combinacion(a, b, alfa=1, beta=1):
    """α a + β b, componente a componente. a y b deben tener la misma cantidad de componentes."""
    a, b = _par(a, b)
    alfa, beta = numero(alfa, "α"), numero(beta, "β")
    return _limpia(alfa * a + beta * b, np.abs(alfa * a) + np.abs(beta * b))


def unitario(v):
    """v / ‖v‖: misma dirección y sentido, magnitud 1. El vector cero no tiene."""
    v = vector(v)
    if not np.any(v):
        raise ValueError("el vector cero no tiene dirección, así que no tiene vector unitario.")
    return v / _norma(v)


def desigualdad_triangulo(a, b, alfa=1, beta=1):
    """(‖α a + β b‖, |α| ‖a‖ + |β| ‖b‖, True si son iguales)."""
    c = combinacion(a, b, alfa, beta)
    a, b = _par(a, b)
    alfa, beta = numero(alfa, "α"), numero(beta, "β")
    izquierda = _norma(c)
    derecha = abs(alfa) * _norma(a) + abs(beta) * _norma(b)
    return izquierda, derecha, abs(derecha - izquierda) <= 1e-12 * derecha


def calculadora_suma(a_txt, b_txt, alfa_txt, beta_txt, n_cifras):
    try:
        a, b = _par(a_txt, b_txt)
        alfa, beta = numero(alfa_txt, "α"), numero(beta_txt, "β")
        c = combinacion(a, b, alfa, beta)
        izq, der, igual = desigualdad_triangulo(a, b, alfa, beta)
    except ValueError as err:
        print("Revisa la entrada:", err); return
    print(f"a = {fmt(a, n_cifras)}    b = {fmt(b, n_cifras)}    α = {cifras(alfa, n_cifras)}    β = {cifras(beta, n_cifras)}")
    print(f"α a + β b = {fmt(c, n_cifras)}")
    print(f"‖α a + β b‖ = {cifras(izq, n_cifras)}")
    try:
        print(f"vector unitario = {fmt(unitario(c), n_cifras)}   (misma dirección y sentido, magnitud 1)")
    except ValueError as err:
        print("vector unitario:", err)
    print(f"‖α a + β b‖ = {cifras(izq, n_cifras)}  ≤  |α| ‖a‖ + |β| ‖b‖ = {cifras(der, n_cifras)}: "
          "se cumple la desigualdad del triángulo")
    print("   y con igualdad: α a y β b apuntan en el mismo sentido (o alguno es el vector cero)." if igual else
          "   con desigualdad estricta: α a y β b no apuntan en el mismo sentido.")
    if len(c) != 2:
        print("(El paralelogramo se dibuja solo para vectores del plano.)")
        return
    with np.errstate(all="ignore"):          # componentes cercanas a 1e100: matplotlib eleva al cuadrado
        _dibuja_paralelogramo(alfa * a, beta * b, c)


def _dibuja_paralelogramo(A, B, c):
    o = np.zeros(2)
    fig, ax = plt.subplots(figsize=(5.5, 4.5))
    for v, color, e in ((A, "tab:red", "α a"), (B, "tab:green", "β b"), (c, "navy", "α a + β b")):
        ax.quiver(*o, *v, angles="xy", scale_units="xy", scale=1, color=color, label=e, width=0.01)
    ax.plot([A[0], c[0]], [A[1], c[1]], "k:", lw=1)       # lados del paralelogramo
    ax.plot([B[0], c[0]], [B[1], c[1]], "k:", lw=1)
    _ejes_iguales(ax, [o, A, B, c])
    ax.grid(True); ax.legend(loc="best", fontsize=8)
    ax.set_xlabel("x"); ax.set_ylabel("y"); ax.set_title("Regla del paralelogramo: α a + β b")
    plt.show()


widgets.interact(calculadora_suma,
    a_txt=widgets.Text(value="300, 0", description="a =", continuous_update=False),
    b_txt=widgets.Text(value="0, 400", description="b =", continuous_update=False),
    alfa_txt=widgets.Text(value="1", description="α =", continuous_update=False),
    beta_txt=widgets.Text(value="1", description="β =", continuous_update=False),
    n_cifras=widgets.IntSlider(value=4, min=2, max=8, description="cifras sig.", continuous_update=False));

In [ ]:
# Casos de prueba de la sección 2 (resultado conocido)
PRUEBAS_2 = [
    ("(300, 0) + (0, 400) = (300, 400), magnitud 500 < 700; resta (300, -400); con α = -1, |α| ‖a‖ + ‖b‖ = 3",
     lambda: np.allclose(combinacion((300, 0), (0, 400), 1, 1), (300, 400))
             and cerca(desigualdad_triangulo((300, 0), (0, 400), 1, 1)[0], 500)
             and not desigualdad_triangulo((300, 0), (0, 400), 1, 1)[2]
             and np.allclose(combinacion((300, 0), (0, 400), 1, -1), (300, -400))
             and cerca(desigualdad_triangulo((1, 0), (2, 0), -1, 1)[1], 3)),
    ("unitario de (2, -1, 2) = (2/3, -1/3, 2/3)", lambda: np.allclose(unitario((2, -1, 2)), (2/3, -1/3, 2/3))),
    ("el vector (0, 0, 0) no tiene unitario", lambda: _rechaza(lambda: unitario((0, 0, 0)))),
    ("(3, 0) + (4, 0): igualdad 7 = 3 + 4",
     lambda: desigualdad_triangulo((3, 0), (4, 0), 1, 1)[2] and cerca(desigualdad_triangulo((3, 0), (4, 0), 1, 1)[0], 7)),
    ("un vector del plano con uno del espacio se rechaza", lambda: _rechaza(lambda: combinacion((1, 2), (1, 2, 3), 1, 1))),
]
for nombre, prueba in PRUEBAS_2:
    print("OK  " if prueba() else "FALLA", nombre)

**Contrasta (sección 2).** Calcula a mano el vector unitario de $(1,-4,8)$: primero su magnitud, después divide cada componente entre ella. Escríbelo como texto, con fracciones o con decimales.

In [ ]:
mi_unitario = None     # escribe tu vector como texto, por ejemplo: "1/3, 2/3, -2/3" o "0.3333, 0.6667, -0.6667"

if mi_unitario is None:     # la referencia se muestra después de tu cálculo
    print("falta tu cálculo: escríbelo arriba y vuelve a ejecutar la celda")
else:
    u_ref = unitario((1, -4, 8))
    exacto = ", ".join(str(Fraction(c).limit_denominator(1000)) for c in u_ref)
    print(f"La calculadora da: ({exacto}) ≈", fmt(u_ref, 4))
    try:
        ok = bool(np.all(np.abs(vector(mi_unitario, 3, "tu vector") - u_ref) < 1e-3))
    except ValueError as err:
        print("Revisa tu valor:", err); ok = None
    if ok is not None:
        print("coinciden" if ok else
              "NO coinciden: la magnitud es la raíz de la suma de los cuadrados; divide cada componente entre ella")

## 3. Producto punto y proyección

El producto punto de dos vectores con la misma cantidad de componentes es un **número**:

$$\mathbf a\cdot\mathbf b=a_1b_1+a_2b_2+\dots+a_nb_n=\|\mathbf a\|\,\|\mathbf b\|\cos\theta,\qquad 0\le\theta\le\pi .$$

* **Ángulo:** $\theta=\arccos\dfrac{\mathbf a\cdot\mathbf b}{\|\mathbf a\|\,\|\mathbf b\|}$, con $\mathbf a,\mathbf b\neq\mathbf 0$. Por redondeo, el cociente de dos vectores paralelos puede salir $1.0000000000000002$, y `arccos` de un número mayor que 1 da `nan`. Por eso la calculadora lo **recorta** al intervalo $[-1,1]$ (`np.clip`) antes de aplicar `arccos`.
* **Signo:** $\mathbf a\cdot\mathbf b>0$, ángulo agudo; $\mathbf a\cdot\mathbf b=0$, recto (los vectores son **ortogonales**); $\mathbf a\cdot\mathbf b<0$, obtuso.
* **Componente y proyección** de $\mathbf a$ sobre $\mathbf b\neq\mathbf 0$, y la parte perpendicular:

$$\mathrm{comp}_{\mathbf b}\,\mathbf a=\frac{\mathbf a\cdot\mathbf b}{\|\mathbf b\|},\qquad \mathrm{proy}_{\mathbf b}\,\mathbf a=\frac{\mathbf a\cdot\mathbf b}{\mathbf b\cdot\mathbf b}\,\mathbf b,\qquad \mathbf a_\perp=\mathbf a-\mathrm{proy}_{\mathbf b}\,\mathbf a .$$

La componente es negativa si el ángulo es obtuso. La comprobación es $\mathbf a_\perp\cdot\mathbf b=0$, y $\mathbf a=\mathrm{proy}_{\mathbf b}\,\mathbf a+\mathbf a_\perp$.

**Similitud coseno.** En ciencia de datos, $\cos\theta$ entre dos vectores de $n$ componentes (por ejemplo, cuántas veces aparece cada palabra en dos documentos) mide qué tan parecidas son sus direcciones sin importar su tamaño: 1 si apuntan igual, 0 si son ortogonales. Por eso la calculadora acepta vectores de 2, 3 o más componentes.

In [ ]:
def punto(a, b):
    """a · b = a1 b1 + a2 b2 + ... (misma cantidad de componentes)."""
    a, b = _par(a, b)
    return _punto(a, b)


def _coseno(a, b):
    """cos θ entre a y b ya leídos, recortado a [-1, 1]. ValueError si alguno es el vector cero."""
    if not np.any(a) or not np.any(b):
        raise ValueError("el vector cero no tiene dirección: no forma ángulo con ningún vector.")
    ea, eb = _escalado(a), _escalado(b)       # escalar por potencias de 2 evita desbordes y no cambia θ
    c = float(np.clip(_punto(ea, eb) / (_norma(ea) * _norma(eb)), -1.0, 1.0))
    # arccos no distingue ángulos menores que ~1e-8 rad: un coseno a 1e-15 de ±1 es de vectores paralelos
    return math.copysign(1.0, c) if abs(c) > 1 - 1e-15 else c


def angulo(a, b, unidad="grados"):
    """Ángulo entre a y b, de 0 a 180° (o de 0 a π rad), con arccos del coseno recortado."""
    a, b = _par(a, b)
    return en_unidad(math.acos(_coseno(a, b)), unidad)


def clasifica(a, b, tol=1e-12):
    """'agudo', 'recto' u 'obtuso' según el signo de a · b (con |cos θ| <= tol se toma como recto)."""
    a, b = _par(a, b)
    c = _coseno(a, b)
    return "recto" if abs(c) <= tol else ("agudo" if c > 0 else "obtuso")


def proyeccion(a, b):
    """(comp_b a, proy_b a, a⊥). b no puede ser el vector cero."""
    a, b = _par(a, b)
    if not np.any(b):
        raise ValueError("no se proyecta sobre el vector cero: b no tiene dirección.")
    ub = b / _norma(b)                        # (a·b/‖b‖) b/‖b‖ = (a·b / b·b) b, sin desbordes
    comp = _punto(a, ub)
    par = comp * ub
    return comp, par, _limpia(a - par, np.abs(a) + np.abs(par))


def calculadora_punto(a_txt, b_txt, unidad, n_cifras):
    try:
        a, b = _par(a_txt, b_txt)
        p = punto(a, b)
    except ValueError as err:
        print("Revisa la entrada:", err); return
    print(f"a = {fmt(a, n_cifras)}    b = {fmt(b, n_cifras)}")
    print(f"a · b = {cifras(p, n_cifras)}")
    try:
        t, clase = angulo(a, b, unidad), clasifica(a, b)
        print(f"cos θ = {cifras(_coseno(a, b), n_cifras)}   (la similitud coseno)")
        signo = {"agudo": "a · b > 0", "obtuso": "a · b < 0",
                 "recto": "a · b = 0" if p == 0 else "a · b ≈ 0"}[clase]
        print(f"θ = {cifras(t, n_cifras)}{_sufijo(unidad)}   →   ángulo {clase} ({signo})")
    except ValueError as err:
        print("ángulo:", err)
    try:
        comp, par, perp = proyeccion(a, b)
    except ValueError as err:
        print("proyección:", err); return
    print(f"comp_b a = {cifras(comp, n_cifras)}")
    print(f"proy_b a = {fmt(par, n_cifras)}   (la parte de a paralela a b)")
    print(f"a⊥ = a − proy_b a = {fmt(perp, n_cifras)}   (la parte de a perpendicular a b)")
    r = punto(perp, b)
    if abs(r) <= 1e-9 * _norma(a) * _norma(b):
        print(f"comprobación: a⊥ · b = {cifras(r, n_cifras) if r else '0'}{' ≈ 0' if r else ''}, "
              "así que a⊥ es perpendicular a b")
    else:
        print(f"comprobación: a⊥ · b = {cifras(r, n_cifras)} ≠ 0: revisa la entrada")


widgets.interact(calculadora_punto,
    a_txt=widgets.Text(value="4, 1, 1", description="a =", continuous_update=False),
    b_txt=widgets.Text(value="2, 2, 1", description="b =", continuous_update=False),
    unidad=widgets.Dropdown(options=["grados", "radianes"], value="grados", description="unidad"),
    n_cifras=widgets.IntSlider(value=4, min=2, max=8, description="cifras sig.", continuous_update=False));

In [ ]:
# Casos de prueba de la sección 3 (resultado conocido)
PRUEBAS_3 = [
    ("(4, 1, 1) · (2, 2, 1) = 11, ángulo 30.20° = 0.5272 rad",
     lambda: cerca(punto((4, 1, 1), (2, 2, 1)), 11) and round(angulo((4, 1, 1), (2, 2, 1), "grados"), 2) == 30.20
             and round(angulo((4, 1, 1), (2, 2, 1), "radianes"), 4) == 0.5272),
    ("clasificación: (1, 2, -1) y (3, 0, 3) recto; (4, 1, 1) y (2, 2, 1) agudo; (1, 1, 0) y (-1, 0, 1) obtuso",
     lambda: clasifica((1, 2, -1), (3, 0, 3)) == "recto" and clasifica((4, 1, 1), (2, 2, 1)) == "agudo"
             and clasifica((1, 1, 0), (-1, 0, 1)) == "obtuso"),
    ("proyección de (3, 1, 2) sobre (1, 2, 2): componente 3, paralela (1, 2, 2), perpendicular (2, -1, 0)",
     lambda: cerca(proyeccion((3, 1, 2), (1, 2, 2))[0], 3)
             and np.allclose(proyeccion((3, 1, 2), (1, 2, 2))[1], (1, 2, 2))
             and np.allclose(proyeccion((3, 1, 2), (1, 2, 2))[2], (2, -1, 0))),
    ("ángulo entre (1, 0) y (0, 1) = π/2 rad", lambda: cerca(angulo((1, 0), (0, 1), "radianes"), math.pi / 2)),
    ("ángulo entre (1, 1, 1) y (2, 2, 2) = 0 (sin nan)", lambda: angulo((1, 1, 1), (2, 2, 2)) == 0),
    ("el ángulo con el vector cero se rechaza", lambda: _rechaza(lambda: angulo((0, 0, 0), (1, 2, 3)))),
]
for nombre, prueba in PRUEBAS_3:
    print("OK  " if prueba() else "FALLA", nombre)

**Contrasta (sección 3).** Antes de calcular, **predice** con el signo de $(2,1,-2)\cdot(3,0,4)$ si el ángulo entre los dos vectores es agudo, recto u obtuso. Después calcula a mano el ángulo y escríbelo en grados o en radianes (indica cuál). Compara con la calculadora en las dos unidades.

In [ ]:
mi_prediccion = None   # "agudo", "recto" u "obtuso", según el signo de a · b
mi_angulo = None       # tu ángulo, por ejemplo: 45.0 o "pi/4"
mi_unidad = "grados"   # "grados" o "radianes": la unidad en que escribiste tu ángulo

if mi_prediccion is None or mi_angulo is None:     # la referencia se muestra después de tu cálculo
    print("falta tu cálculo: escríbelo arriba y vuelve a ejecutar la celda")
else:
    a_c, b_c = (2, 1, -2), (3, 0, 4)
    print(f"La calculadora da: a · b = {cifras(punto(a_c, b_c), 4)}, ángulo {clasifica(a_c, b_c)}, "
          f"θ = {cifras(angulo(a_c, b_c, 'grados'), 5)}° = {cifras(angulo(a_c, b_c, 'radianes'), 5)} rad")
    pred = str(mi_prediccion).strip(" .").lower()
    print("predicción: correcta" if pred == clasifica(a_c, b_c) else
          "predicción: NO coincide; el signo de a · b decide: positivo agudo, cero recto, negativo obtuso")
    unidad_c = str(mi_unidad).strip().lower()
    otra = {"grados": "radianes", "radianes": "grados"}
    try:
        texto = mi_angulo.replace("°", "").replace("º", "") if isinstance(mi_angulo, str) else mi_angulo
        valor = numero(texto, "tu ángulo")
        ok = cerca(math.degrees(de_unidad(valor, unidad_c)), angulo(a_c, b_c, "grados"), 0.05)   # tolerancia: 0.05°
        if not ok and cerca(math.degrees(de_unidad(valor, otra[unidad_c])), angulo(a_c, b_c, "grados"), 0.05):
            print(f"Tu número está bien, pero en {otra[unidad_c]}: cambia mi_unidad a \"{otra[unidad_c]}\".")
    except ValueError as err:
        print("Revisa tu valor:", err); ok = None
    if ok is not None:
        print("ángulo: coinciden" if ok else
              "ángulo: NO coinciden; θ = arccos(a · b / (‖a‖ ‖b‖)), con la calculadora en la unidad que escribiste")

## 4. Producto cruz

El producto cruz combina dos vectores **de 3 componentes** en un tercer vector (definición 1.13 del libro):

$$\mathbf a\times\mathbf b=\big(a_yb_z-a_zb_y,\;\;a_zb_x-a_xb_z,\;\;a_xb_y-a_yb_x\big).$$

* Es **perpendicular** a los dos factores: $(\mathbf a\times\mathbf b)\cdot\mathbf a=0$ y $(\mathbf a\times\mathbf b)\cdot\mathbf b=0$. Su sentido lo da la regla de la mano derecha.
* Su magnitud es el **área** del paralelogramo que forman: $\|\mathbf a\times\mathbf b\|=\|\mathbf a\|\,\|\mathbf b\|\,\mathrm{sen}\,\theta$. La mitad es el área del triángulo.
* **El orden importa:** $\mathbf b\times\mathbf a=-(\mathbf a\times\mathbf b)$ (anticonmutatividad). Un momento $\mathbf r\times\mathbf F$ escrito al revés apunta en sentido contrario.
* $\mathbf a\times\mathbf b=\mathbf 0$ si los vectores son **paralelos** (o alguno es el vector cero): no encierran área.
* Solo existe en 3D. Si das dos vectores del plano, la calculadora les agrega $z=0$ y te avisa; el resultado queda sobre el eje $z$.

In [ ]:
def _cruz(a, b):
    """a × b de dos vectores de 3 componentes ya leídos, con la cancelación por redondeo limpiada."""
    terminos = np.array([[a[1] * b[2], a[2] * b[1]],
                         [a[2] * b[0], a[0] * b[2]],
                         [a[0] * b[1], a[1] * b[0]]])
    return _limpia(terminos[:, 0] - terminos[:, 1], np.abs(terminos).sum(axis=1))


def cruz(a, b):
    """(a × b, aviso). aviso es '' en 3D o un mensaje si a vectores del plano se les agregó z = 0."""
    a, b = _par(a, b)
    aviso = ""
    if len(a) == 2:
        a, b = np.append(a, 0.0), np.append(b, 0.0)
        aviso = "Aviso: el producto cruz solo existe en 3D; a los vectores del plano se les agregó z = 0."
    elif len(a) != 3:
        raise ValueError(f"el producto cruz solo está definido para vectores de 3 componentes "
                         f"(o del plano, agregando z = 0); estos tienen {len(a)}.")
    return _cruz(a, b), aviso


def paralelos(a, b, tol=1e-12):
    """True si a × b = 0 dentro del redondeo: mismo sentido, sentido opuesto, o alguno es el vector cero."""
    a, b = _par(a, b)
    if not np.any(a) or not np.any(b):
        return True
    ea, eb = _escalado(a), _escalado(b)       # escalar por potencias de 2 no cambia la dirección
    return _norma(cruz(ea, eb)[0]) <= tol * _norma(ea) * _norma(eb)


def calculadora_cruz(a_txt, b_txt, n_cifras):
    try:
        a, b = _par(a_txt, b_txt)
        c, aviso = cruz(a, b)
        d, _ = cruz(b, a)
    except ValueError as err:
        print("Revisa la entrada:", err); return
    if aviso:
        print(aviso)
        a, b = np.append(a, 0.0), np.append(b, 0.0)
    area = _norma(c)
    print(f"a × b = {fmt(a, n_cifras)} × {fmt(b, n_cifras)} = {fmt(c, n_cifras)}")
    print(f"b × a = {fmt(b, n_cifras)} × {fmt(a, n_cifras)} = {fmt(d, n_cifras)}   (el orden importa: b × a = −(a × b))")
    print(f"área del paralelogramo = ‖a × b‖ = {cifras(area, n_cifras)}")
    print(f"área del triángulo = ‖a × b‖ / 2 = {cifras(area / 2, n_cifras)}")
    print(f"comprobación: (a × b) · a = {cifras(punto(c, a), n_cifras)}   y   (a × b) · b = {cifras(punto(c, b), n_cifras)}",
          end="")
    if paralelos(a, b):
        print("\na × b = 0: los vectores son paralelos (o alguno es el vector cero) y no encierran área.")
    else:
        print("   (a × b es perpendicular a los dos)")
    with np.errstate(all="ignore"):          # componentes cercanas a 1e100: matplotlib eleva al cuadrado
        _dibuja_cruz(a, b, c, area)


def _dibuja_cruz(a, b, c, area):
    o = np.zeros(3)
    fig = plt.figure(figsize=(6, 5.5))
    ax = fig.add_subplot(projection="3d")
    for v, color, e in ((a, "tab:red", "a"), (b, "tab:green", "b"), (c, "navy", "a × b")):
        ax.quiver(*o, *v, color=color, arrow_length_ratio=0.1, lw=2, label=e)
    ax.plot(*np.array([a, a + b, b]).T, "k:", lw=1, label="paralelogramo")
    _ejes_iguales(ax, [o, a, b, a + b, c])
    _vista(ax, [a, b], normal=c)               # ninguna flecha apunta al ojo; el paralelogramo, inclinado
    ax.set_xlabel("x"); ax.set_ylabel("y"); ax.set_zlabel("z")
    ax.set_title(f"a × b = {fmt(c, 3)}, área = {cifras(area, 3)}")
    ax.legend(loc="upper left", fontsize=8)
    plt.show()


widgets.interact(calculadora_cruz,
    a_txt=widgets.Text(value="1, 2, 3", description="a =", continuous_update=False),
    b_txt=widgets.Text(value="4, 5, 6", description="b =", continuous_update=False),
    n_cifras=widgets.IntSlider(value=4, min=2, max=8, description="cifras sig.", continuous_update=False));

In [ ]:
# Casos de prueba de la sección 4 (resultado conocido)
PRUEBAS_4 = [
    ("(1, 2, 3) × (4, 5, 6) = (-3, 6, -3), en orden invertido (3, -6, 3); no son paralelos",
     lambda: np.allclose(cruz((1, 2, 3), (4, 5, 6))[0], (-3, 6, -3)) and np.allclose(cruz((4, 5, 6), (1, 2, 3))[0], (3, -6, 3))
             and not paralelos((1, 2, 3), (4, 5, 6))),
    ("i × j = k y j × i = -k",
     lambda: np.allclose(cruz((1, 0, 0), (0, 1, 0))[0], (0, 0, 1)) and np.allclose(cruz((0, 1, 0), (1, 0, 0))[0], (0, 0, -1))),
    ("(1, -3, 2) × (-2, 6, -4) = 0: son paralelos",
     lambda: np.allclose(cruz((1, -3, 2), (-2, 6, -4))[0], 0) and paralelos((1, -3, 2), (-2, 6, -4))),
    ("(3, 0) × (0, 4) = (0, 0, 12), con aviso de z = 0",
     lambda: np.allclose(cruz((3, 0), (0, 4))[0], (0, 0, 12)) and cruz((3, 0), (0, 4))[1] != ""),
    ("vectores de 4 componentes se rechazan", lambda: _rechaza(lambda: cruz((1, 2, 3, 4), (5, 6, 7, 8)))),
]
for nombre, prueba in PRUEBAS_4:
    print("OK  " if prueba() else "FALLA", nombre)

**Contrasta (sección 4).** Calcula a mano $(2,-1,0)\times(1,3,2)$ con la fórmula de la definición 1.13 y escribe el resultado. Después invierte el orden y calcula $(1,3,2)\times(2,-1,0)$.

In [ ]:
mi_axb = None     # tu a × b como texto, por ejemplo: "1, -2, 3"
mi_bxa = None     # después, b × a (el orden invertido)

a_c, b_c = (2, -1, 0), (1, 3, 2)
if mi_axb is None:     # la referencia se muestra después de tu cálculo
    print("falta tu cálculo: escríbelo arriba y vuelve a ejecutar la celda")
else:
    print("La calculadora da: a × b =", fmt(cruz(a_c, b_c)[0], 4))
    try:
        ok = np.allclose(vector(mi_axb, 3, "tu a × b"), cruz(a_c, b_c)[0], rtol=0, atol=1e-6)
    except ValueError as err:
        print("Revisa tu valor:", err); ok = None
    if ok is not None:
        print("a × b: coinciden" if ok else
              "a × b: NO coinciden; usa (ay bz − az by, az bx − ax bz, ax by − ay bx) y cuida los signos")
    if mi_bxa is None:
        print("falta tu b × a: escríbelo arriba y vuelve a ejecutar la celda")
    else:
        print("La calculadora da: b × a =", fmt(cruz(b_c, a_c)[0], 4))
        try:
            ok = np.allclose(vector(mi_bxa, 3, "tu b × a"), cruz(b_c, a_c)[0], rtol=0, atol=1e-6)
        except ValueError as err:
            print("Revisa tu valor:", err); ok = None
        if ok is not None:
            print("b × a: coinciden; es −(a × b)" if ok else
                  "b × a: NO coinciden; al invertir el orden, cada componente cambia de signo")

## 5. Rectas y planos en el espacio

* **Recta** por $P_0(x_0,y_0,z_0)$ con dirección $\mathbf u=(a,b,c)\neq\mathbf 0$: ecuación vectorial $\mathbf r(t)=P_0+t\,\mathbf u$ y paramétricas $x=x_0+at,\;y=y_0+bt,\;z=z_0+ct$, con $t$ real.
* **Plano** por $P_0$ con normal $\mathbf n=(a,b,c)\neq\mathbf 0$: los puntos $P$ con $\mathbf n\cdot(P-P_0)=0$. Al desarrollar queda la ecuación cartesiana $ax+by+cz=d$, con $d=\mathbf n\cdot P_0$.
* **Plano por tres puntos** $A$, $B$, $C$: una normal es $\mathbf n=\overrightarrow{AB}\times\overrightarrow{AC}$. Si sale $\mathbf 0$, los puntos están alineados y no definen un plano único.
* **Distancia** de un punto $Q$ al plano: $D=\dfrac{|\mathbf n\cdot Q-d|}{\|\mathbf n\|}$. El signo de $\mathbf n\cdot Q-d$ dice de qué lado queda $Q$: positivo, del lado hacia donde apunta $\mathbf n$.
* **Intersección** de la recta con el plano: al sustituir $\mathbf r(t)$ queda $\mathbf n\cdot P_0+t\,(\mathbf n\cdot\mathbf u)=d$.

| Caso | Condición | Resultado |
|---|---|---|
| corta en un punto | $\mathbf n\cdot\mathbf u\neq0$ | $t^*=\dfrac{d-\mathbf n\cdot P_0}{\mathbf n\cdot\mathbf u}$, en el punto $P_0+t^*\mathbf u$ |
| paralela | $\mathbf n\cdot\mathbf u=0$ y $\mathbf n\cdot P_0\neq d$ | no toca el plano |
| contenida | $\mathbf n\cdot\mathbf u=0$ y $\mathbf n\cdot P_0=d$ | toda la recta está en el plano |

Cualquier múltiplo no nulo de $\mathbf n$ (con $d$ multiplicado por lo mismo) describe el mismo plano. Con tres puntos, la calculadora simplifica: si las coordenadas son enteras o fracciones (como `7/5` o `0.25`), hace las cuentas con fracciones exactas y da coeficientes enteros sin factor común; si no (por ejemplo, con `sqrt(2)`), los divide entre el de mayor valor absoluto. Además deja positivo el primer coeficiente que no es cero. Si $t^*<0$, el corte queda antes de $P_0$: un rayo (solo $t\ge0$), como el de una cámara, no llega a él.

In [ ]:
def _normal(n):
    n = vector(n, 3, "la normal n")
    if not np.any(n):
        raise ValueError("la normal n no puede ser el vector cero.")
    return n


def _enteros(valores):
    """True si todos son enteros (dentro del redondeo). 1e-200 no cuenta como el entero 0."""
    return all(abs(v) < 2**53 and abs(v - round(v)) <= 1e-9 * max(1.0, abs(v)) and (v == 0 or round(v) != 0)
               for v in valores)


def _num_txt(v, n=4):
    """Entero exacto sin decimales; lo demás, redondeado a n cifras significativas y sin ceros de relleno que
    parezcan exactos: 123456789.4 con 4 cifras se escribe 1.235×10^8, no 123500000."""
    if _enteros([v]):
        return str(int(round(v)))
    texto = cifras(v, n)
    if "." in texto or "×" in texto or len(texto.lstrip("-")) <= n:
        return texto
    q = Decimal(texto)                            # entero con ceros de relleno: a notación científica
    e = q.adjusted()
    return f"{format(q.scaleb(-e).quantize(Decimal(1).scaleb(1 - n)), 'f')}×10^{e}"


def _fmt_txt(v, n=4):
    """Vector '(a, b, c)' con el mismo formato que los coeficientes del plano."""
    return "(" + ", ".join(_num_txt(c, n) for c in v) + ")"


def _lado(P, n, d):
    """n · P − d, con la cancelación por redondeo limpiada."""
    terminos = [*(n * P), -d]
    s = math.fsum(terminos)
    return 0.0 if abs(s) <= 1e-14 * math.fsum(abs(x) for x in terminos) else s


def _fraccion(x):
    """x como fracción de denominador hasta 10^6 si x es exactamente el flotante de esa fracción (7/5, 0.25,
    -9/7); si no (sqrt(2), pi), None."""
    f = Fraction(x).limit_denominator(10**6)
    return f if float(f) == x else None


def _plano_exacto(A, B, C):
    """(n, d) con enteros sin factor común, calculados con fracciones exactas; None si alguna coordenada
    no es una fracción sencilla o los enteros no caben en un flotante."""
    fr = [_fraccion(x) for x in (*A, *B, *C)]
    if any(f is None for f in fr):
        return None
    a, b, c = fr[0:3], fr[3:6], fr[6:9]
    u = [q - p for p, q in zip(a, b)]
    w = [q - p for p, q in zip(a, c)]
    n = [u[1] * w[2] - u[2] * w[1], u[2] * w[0] - u[0] * w[2], u[0] * w[1] - u[1] * w[0]]
    if not any(n):
        return None
    d = sum(ni * ai for ni, ai in zip(n, a))
    m = math.lcm(*(f.denominator for f in (*n, d)))
    enteros = [int(f * m) for f in (*n, d)]
    g = math.gcd(*enteros)
    try:
        return np.array([float(e // g) for e in enteros[:3]]), float(enteros[3] // g)
    except OverflowError:
        return None


def plano_tres_puntos(A, B, C):
    """(n, d) del plano n · (x, y, z) = d que pasa por A, B y C, con n = AB × AC simplificado: enteros sin
    factor común si las coordenadas son enteras o fracciones sencillas; si no, divididos entre el coeficiente
    de mayor valor absoluto. El primer coeficiente distinto de cero queda positivo."""
    A, B, C = vector(A, 3, "A"), vector(B, 3, "B"), vector(C, 3, "C")
    AB, AC = B - A, C - A
    if paralelos(AB, AC):
        raise ValueError("los puntos están alineados (o repetidos): no definen un plano único.")
    exacto = _plano_exacto(A, B, C)
    if exacto is not None:
        n, d = exacto
    else:
        n, _ = cruz(AB, AC)
        d = _punto(n, A)
        escala = float(np.max(np.abs(n)))
        n, d = n / escala, d / escala
    if n[np.flatnonzero(n)[0]] < 0:
        n, d = -n, -d
    return n + 0.0, float(d) + 0.0


def plano_texto(n, d, n_cifras=4):
    """'3x + 5y + 4z = 13' a partir de la normal n y de d."""
    n, d = _normal(n), numero(d, "d")
    texto = ""
    for c, var in zip(n, "xyz"):
        if c == 0:
            continue
        coef = _num_txt(abs(c), n_cifras)
        termino = ("" if coef == "1" else coef) + var
        texto += (("-" if c < 0 else "") + termino) if not texto else ((" - " if c < 0 else " + ") + termino)
    return f"{texto} = {_num_txt(d, n_cifras)}"


def distancia(P, n, d):
    """(D, lado): distancia del punto P al plano n · (x, y, z) = d y de qué lado queda."""
    P, n, d = vector(P, 3, "el punto"), _normal(n), numero(d, "d")
    s = _lado(P, n, d)
    lado = ("sobre el plano" if s == 0 else
            "del lado hacia donde apunta n" if s > 0 else "del lado contrario a n")
    return abs(s) / _norma(n), lado


def interseccion(P0, u, n, d):
    """('punto', t*, X), ('paralela', None, None) o ('contenida', None, None) para la recta P0 + t u."""
    P0, u, n, d = vector(P0, 3, "P0"), vector(u, 3, "la dirección u"), _normal(n), numero(d, "d")
    if not np.any(u):
        raise ValueError("la dirección u de la recta no puede ser el vector cero.")
    s = _lado(P0, n, d)
    den = _punto(n, u)
    if abs(den) <= 1e-12 * _norma(n) * _norma(u):          # n · u = 0: la recta va paralela al plano
        return ("contenida", None, None) if s == 0 else ("paralela", None, None)
    t = -s / den + 0.0
    return "punto", t, _limpia(P0 + t * u, np.abs(P0) + np.abs(t * u))


def misma_ecuacion(p, q, tol=1e-9):
    """True si (a, b, c, d) y (a', b', c', d') son múltiplos no nulos uno del otro (el mismo plano)."""
    p, q = vector(p, 4, "la ecuación (a, b, c, d)"), vector(q, 4, "la ecuación (a, b, c, d)")
    if not np.any(p[:3]) or not np.any(q[:3]):
        raise ValueError("a, b y c no pueden ser los tres 0: la normal sería el vector cero.")
    up, uq = p / _norma(p), q / _norma(q)
    return min(_norma(up - uq), _norma(up + uq)) <= tol


def _parametricas(P0, u, n_cifras):
    """'x = 1 + 2t,   y = -t,   z = 3' (sin términos que valen 0)."""
    partes = []
    for x0, a, var in zip(P0, u, "xyz"):
        coef = _num_txt(abs(a), n_cifras)
        termino = ("" if coef == "1" else coef) + "t"
        if a == 0:
            partes.append(f"{var} = {_num_txt(x0, n_cifras)}")
        elif x0 == 0:
            partes.append(f"{var} = {'-' if a < 0 else ''}{termino}")
        else:
            partes.append(f"{var} = {_num_txt(x0, n_cifras)} {'-' if a < 0 else '+'} {termino}")
    return ",   ".join(partes)


def _pie(P, n, d):
    """Punto del plano más cercano a P (pie de la perpendicular), con la normal unitaria: sin desbordes."""
    nn = _norma(n)
    return P - (_lado(P, n, d) / nn) * (n / nn)


def _base_plano(n):
    """Dos vectores unitarios perpendiculares entre sí y a n (para dibujar el plano)."""
    nu = n / _norma(n)
    e1 = np.cross(nu, np.eye(3)[np.argmin(np.abs(nu))])
    e1 = e1 / _norma(e1)
    return e1, np.cross(nu, e1)


def calculadora_recta_plano(modo, A_txt, B_txt, C_txt, n_txt, d_txt, P0_txt, u_txt, Q_txt, n_cifras):
    try:
        if modo == "plano por tres puntos":
            pts = [vector(t, 3, e) for t, e in ((A_txt, "A"), (B_txt, "B"), (C_txt, "C"))]
            n, d = plano_tres_puntos(*pts)
        else:
            pts = []
            n, d = _normal(n_txt), numero(d_txt, "d")
    except ValueError as err:
        print("Revisa la entrada:", err); return
    print(f"Plano: {plano_texto(n, d, n_cifras)}      (normal n = {fmt(n, n_cifras)}, d = {_num_txt(d, n_cifras)})")
    if pts:
        print("   n = AB × AC simplificado. Comprobación: n · A, n · B, n · C =",
              ", ".join(_num_txt(_punto(n, P), n_cifras) for P in pts), " (las tres deben dar d)")
        print("   (en este modo no se usan los campos n y d)")
    else:
        print("   (en este modo no se usan los campos A, B y C)")

    Q = H = P0 = X = None
    try:
        Q = vector(Q_txt, 3, "Q")
        D, lado = distancia(Q, n, d)
        print(f"Distancia de Q = {fmt(Q, n_cifras)} al plano: D = {_num_txt(D, n_cifras)}; Q queda {lado}.")
        H = _pie(Q, n, d)                               # pie de la perpendicular, para el dibujo
    except ValueError as err:
        print("Revisa la entrada:", err); Q = None
    try:
        P0, u = vector(P0_txt, 3, "P0"), vector(u_txt, 3, "la dirección u")
        tipo, t, X = interseccion(P0, u, n, d)
    except ValueError as err:
        print("Revisa la entrada:", err); P0 = None
    if P0 is not None:
        print(f"Recta: r(t) = P0 + t u = {fmt(P0, n_cifras)} + t {fmt(u, n_cifras)}")
        print("   paramétricas:", _parametricas(P0, u, n_cifras))
        if tipo == "punto":
            print(f"La recta corta al plano en t* = {_num_txt(t, n_cifras)}, en el punto {_fmt_txt(X, n_cifras)}.")
            if t < 0:
                print("t* < 0: el corte queda antes de P0 (detrás, si la recta es un rayo)")
        elif tipo == "paralela":
            print("La recta es paralela al plano y no lo toca: n · u = 0 y n · P0 ≠ d.")
        else:
            print("La recta está contenida en el plano: n · u = 0 y n · P0 = d.")

    # Figura: un trozo del plano centrado en lo que interesa, la recta, el corte y la distancia
    interes = pts + [p for p in (Q, H, P0, X) if p is not None]
    if not interes:
        interes = [_pie(np.zeros(3), n, d)]
    centro = _pie(np.mean(interes, axis=0), n, d)
    R = 1.2 * max(_norma(p - centro) for p in interes) or 1.0
    e1, e2 = _base_plano(n)
    s1, s2 = np.meshgrid([-R, R], [-R, R])
    malla = [centro[i] + s1 * e1[i] + s2 * e2[i] for i in range(3)]
    fig = plt.figure(figsize=(6.5, 6))
    ax = fig.add_subplot(projection="3d")
    ax.plot_surface(*malla, color="tab:blue", alpha=0.25, shade=False)
    ax.quiver(*centro, *(0.5 * R * n / _norma(n)), color="tab:blue", arrow_length_ratio=0.15, label="dirección de n")
    dibujados = [centro, *(np.array(malla).reshape(3, -1).T)]
    for P, e in zip(pts, "ABC"):
        ax.scatter(*P, color="tab:blue", s=15); ax.text(*P, f"  {e}")
    if P0 is not None:
        L = R / _norma(u)
        t0, t1 = (min(0.0, t), max(0.0, t)) if tipo == "punto" else (0.0, 0.0)
        extremos = np.array([P0 + (t0 - L) * u, P0 + (t1 + L) * u])
        ax.plot(*extremos.T, color="black", lw=1.5, label="recta P0 + t u")
        mismo = Q is not None and np.array_equal(P0, Q)
        ax.scatter(*P0, color="black", s=20); ax.text(*P0, "  P0 = Q" if mismo else "  P0")
        dibujados += list(extremos)
        if tipo == "punto":
            ax.scatter(*X, color="red", s=45, label=f"corte (t* = {_num_txt(t, 3)})")
    if Q is not None:
        ax.scatter(*Q, color="green", s=20)
        if P0 is None or not np.array_equal(P0, Q):
            ax.text(*Q, "  Q")
        ax.plot(*np.array([Q, H]).T, ":", color="green", lw=1.5, label=f"distancia D = {_num_txt(D, 3)}")
    _ejes_iguales(ax, dibujados + interes)
    _vista(ax, [u] if P0 is not None else [], normal=n)   # el plano no se ve de canto ni de frente
    ax.set_xlabel("x"); ax.set_ylabel("y"); ax.set_zlabel("z")
    ax.set_title(f"Plano {plano_texto(n, d, 3)}")
    ax.legend(loc="upper left", fontsize=8)
    plt.show()


_ancho = {"description_width": "initial"}
widgets.interact(calculadora_recta_plano,
    modo=widgets.Dropdown(options=["plano por tres puntos", "plano por normal y d"], value="plano por tres puntos",
                          description="modo"),
    A_txt=widgets.Text(value="1, 2, 0", description="A =", continuous_update=False),
    B_txt=widgets.Text(value="3, 0, 1", description="B =", continuous_update=False),
    C_txt=widgets.Text(value="0, 1, 2", description="C =", continuous_update=False),
    n_txt=widgets.Text(value="2, 1, 2", description="n =", continuous_update=False),
    d_txt=widgets.Text(value="12", description="d =", continuous_update=False),
    P0_txt=widgets.Text(value="0, 0, 0", description="P0 (recta) =", continuous_update=False, style=_ancho),
    u_txt=widgets.Text(value="1, 1, 1", description="u (recta) =", continuous_update=False, style=_ancho),
    Q_txt=widgets.Text(value="2, 3, 4", description="Q (distancia) =", continuous_update=False, style=_ancho),
    n_cifras=widgets.IntSlider(value=4, min=2, max=8, description="cifras sig.", continuous_update=False));

In [ ]:
# Casos de prueba de la sección 5 (resultado conocido)
def _plano_libro():
    """El plano por (1, 2, 0), (3, 0, 1) y (0, 1, 2) tiene n ∥ (3, 5, 4) y pasa por los tres puntos."""
    puntos = ((1, 2, 0), (3, 0, 1), (0, 1, 2))
    n, d = plano_tres_puntos(*puntos)
    return (paralelos(n, (3, 5, 4)) and all(cerca(punto(n, P), d) for P in puntos)
            and plano_texto(n, d) == "3x + 5y + 4z = 13"
            and misma_ecuacion((6, 10, 8, 26), (3, 5, 4, 13)) and not misma_ecuacion((3, 5, 4, 12), (3, 5, 4, 13)))


PRUEBAS_5 = [
    ("plano por (1, 2, 0), (3, 0, 1), (0, 1, 2): 3x + 5y + 4z = 13, pasa por los tres; (6, 10, 8, 26) es el mismo",
     _plano_libro),
    ("distancia de (4, 5, 6) a x + 2y + 2z = 12 = 14/3, del lado hacia donde apunta n",
     lambda: cerca(distancia((4, 5, 6), (1, 2, 2), 12)[0], 14 / 3)
             and distancia((4, 5, 6), (1, 2, 2), 12)[1] == "del lado hacia donde apunta n"),
    ("recta (0, 0, 1) + t (1, 1, 1) y plano 2x + y + 2z = 12: corta en t = 2, en (2, 2, 3)",
     lambda: interseccion((0, 0, 1), (1, 1, 1), (2, 1, 2), 12)[0] == "punto"
             and cerca(interseccion((0, 0, 1), (1, 1, 1), (2, 1, 2), 12)[1], 2)
             and np.allclose(interseccion((0, 0, 1), (1, 1, 1), (2, 1, 2), 12)[2], (2, 2, 3))),
    ("con u = (1, 0, -1) la recta es paralela al plano",
     lambda: interseccion((0, 0, 1), (1, 0, -1), (2, 1, 2), 12)[0] == "paralela"),
    ("los puntos alineados (1, 2, 3), (3, 5, 4), (5, 8, 5) se rechazan",
     lambda: _rechaza(lambda: plano_tres_puntos((1, 2, 3), (3, 5, 4), (5, 8, 5)))),
    ("recta (1, 1, 0) + t (1, 0, 0) contenida en el plano z = 0",
     lambda: interseccion((1, 1, 0), (1, 0, 0), (0, 0, 1), 0)[0] == "contenida"),
]
for nombre, prueba in PRUEBAS_5:
    print("OK  " if prueba() else "FALLA", nombre)

**Contrasta (sección 5).** Encuentra a mano la ecuación $ax+by+cz=d$ del plano que pasa por $(2,0,0)$, $(0,4,0)$ y $(0,0,4)$. Escribe los cuatro números $a, b, c, d$. Tu ecuación puede diferir de la de la calculadora en un factor: cualquier múltiplo no nulo describe el mismo plano.

In [ ]:
mi_plano = None     # escribe a, b, c, d de tu ecuación ax + by + cz = d, por ejemplo: "1, -1, 2, 5"

if mi_plano is None:     # la referencia se muestra después de tu cálculo
    print("falta tu cálculo: escríbelo arriba y vuelve a ejecutar la celda")
else:
    n_ref, d_ref = plano_tres_puntos((2, 0, 0), (0, 4, 0), (0, 0, 4))
    print("La calculadora da:", plano_texto(n_ref, d_ref))
    try:
        ok = misma_ecuacion(mi_plano, (*n_ref, d_ref), tol=1e-3)
    except ValueError as err:
        if isinstance(mi_plano, str) and re.search(r"[xyz=]", mi_plano):
            err = "escribe solo los cuatro números a, b, c, d; por ejemplo, para 2x + 3y - z = 5 escribe \"2, 3, -1, 5\"."
        print("Revisa tu valor:", err); ok = None
    if ok is not None:
        print("coinciden: describen el mismo plano" if ok else
              "NO coinciden; usa n = AB × AC, d = n · A, y comprueba que los tres puntos cumplan tu ecuación")

## Verificación final
Ejecuta esta celda al terminar. Debe imprimir que todas las pruebas pasaron.

In [ ]:
todas = PRUEBAS_1 + PRUEBAS_2 + PRUEBAS_3 + PRUEBAS_4 + PRUEBAS_5
fallas = [nombre for nombre, prueba in todas if not prueba()]
assert not fallas, f"Fallan {len(fallas)} pruebas: {fallas}"
print(f"Todas las verificaciones pasaron: {len(todas)}/{len(todas)}")